# Нейронные сети и глубокое обучение
## Лабораторная работа №2. Свёрточные нейронные сети.

### 1. Загрузка данных

In [1]:
from keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Обучающие изображения:", x_train.shape)
print("Обучающие метки:", y_train.shape)
print("Тестовые изображения:", x_test.shape)
print("Тестовые метки:", y_test.shape)


150118400/170498071 ━━━━━━━━━━━━━━━━━━━━ 5:19 16us/step

Exception: URL fetch failure on https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz: None -- retrieval incomplete: got only 150112000 out of 170498071 bytes

**50 000** - количество обучающих изображений.

**32 × 32** - высота и ширина каждого изображения в пикселях.

**3** - цветовые каналы: RGB.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

fig, axes = plt.subplots(10, 3, figsize=(15, 30))

for class_id in range(10):
    # Находим индексы изображений текущего класса
    indices = np.where(y_train.ravel() == class_id)[0][:3]

    for column, index in enumerate(indices):
        axes[class_id, column].imshow(x_train[index])
        axes[class_id, column].set_title(class_names[class_id])
        axes[class_id, column].axis("off")

plt.tight_layout()
plt.show()

### 2. Построение модели

In [ ]:
from sklearn.model_selection import train_test_split

# Превращаем метки в одномерные массивы
y_train = y_train.ravel()
y_test = y_test.ravel()

x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

**Train — 40 000:** для настройки весов.

**Validation — 10 000:** для контроля обучения и выбора лучших весов.

**Test — 10 000:** для итоговой проверки.

In [ ]:
from keras import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input, Rescaling

model = Sequential([
    Input(shape=(32, 32, 3)),
    
    Rescaling(1. / 255), # Нормализация пикселей в диапазон [0, 1]

    Conv2D(32, (3, 3), activation='relu', padding='same'), # выделяем 32 фильтра размером 3x3, применяем функцию активации ReLU и padding='same' для сохранения размеров изображения
    MaxPooling2D(pool_size=(2, 2)), # после свёртки уменьшает высоту и ширину вдвое: 32 -> 16 

    Conv2D(64, (3, 3), activation='relu', padding='same'), # выделяем 64 фильтра размером 3x3, применяем функцию активации ReLU и padding='same' для сохранения размеров изображения
    MaxPooling2D(pool_size=(2, 2)), # после свёртки уменьшает высоту и ширину вдвое: 16 -> 8

    Conv2D(128, (3, 3), activation='relu', padding='same'), # выделяем 128 фильтров размером 3x3, применяем функцию активации ReLU и padding='same' для сохранения размеров изображения
    MaxPooling2D(pool_size=(2, 2)), # после свёртки уменьшает высоту и ширину вдвое: 8 -> 4

    Flatten(name='flatten'),  # преобразуем 3D-тензор в 1D-вектор

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(10, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### 3. Обучение 

In [ ]:
from keras.callbacks import ModelCheckpoint, CSVLogger

checkpoint = ModelCheckpoint(
    filepath="best_cnn.weights.h5",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

logger = CSVLogger("cnn_training.csv")

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[checkpoint, logger]
)

# Восстанавливаем лучшие веса и сохраняем модель
model.load_weights("best_cnn.weights.h5")
model.save("cnn_cifar10.keras")

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Точность
axes[0].plot(epochs, history.history["accuracy"], label="Train")
axes[0].plot(epochs, history.history["val_accuracy"], label="Validation")
axes[0].set_title("Точность")
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid()

# Потери
axes[1].plot(epochs, history.history["loss"], label="Train")
axes[1].plot(epochs, history.history["val_loss"], label="Validation")
axes[1].set_title("Потери")
axes[1].set_xlabel("Эпоха")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
best_epoch = np.argmax(history.history["val_accuracy"]) + 1
best_val_accuracy = max(history.history["val_accuracy"])

print(f"Лучшая эпоха по val_accuracy: {best_epoch}")
print(f"Лучшая валидационная точность: {best_val_accuracy:.2%}")

model.load_weights("best_cnn.weights.h5")

test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    batch_size=64,
    verbose=0
)

print(f"Тестовая потеря: {test_loss:.4f}")
print(f"Тестовая точность: {test_accuracy:.2%}")

In [ ]:
import pandas as pd

# Получаем вероятности классов для всех тестовых картинок
probabilities = model.predict(x_test, batch_size=64)

# Выбираем класс с максимальной вероятностью
pred10 = np.argmax(probabilities, axis=1)

accuracy_10 = []

for class_id in range(10):
    mask = y_test == class_id

    class_accuracy = np.mean(pred10[mask] == y_test[mask]) * 100

    accuracy_10.append(class_accuracy)

results = pd.DataFrame({
    "Класс": class_names,
    "Ini_10class": accuracy_10
})

display(results.round(2))

results.to_csv(
    "comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
fig, axes = plt.subplots(10, 3, figsize=(15, 30))

for class_id in range(10):
    # Изображения этого класса, распознанные неправильно
    error_indices = np.where((y_test == class_id) & (pred10 != y_test))[0][:3]

    for column, ax in enumerate(axes[class_id]):
        ax.axis("off")

        if column < len(error_indices):
            index = error_indices[column]

            true_name = class_names[class_id]
            predicted_name = class_names[pred10[index]]

            ax.imshow(x_test[index])
            ax.set_title(f"True: {true_name}\nPredicted: {predicted_name}")

plt.tight_layout()
plt.savefig("cnn_errors.png", dpi=150, bbox_inches="tight")
plt.show()

### 4. Обучение на 8 классах и дообучение на 10 классах.

In [ ]:
excluded_classes = [3, 8]

# Исходные номера оставшихся классов
kept_classes = [
    i for i in range(10)
    if i not in excluded_classes
]

# Отбираем изображения оставшихся классов
train_mask = np.isin(y_train, kept_classes)
val_mask = np.isin(y_val, kept_classes)

x_train8 = x_train[train_mask]
x_val8 = x_val[val_mask]

# Соответствие: исходная метка → новая метка от 0 до 7
label_mapping = {
    old_label: new_label
    for new_label, old_label in enumerate(kept_classes)
}

y_train8 = np.array([
    label_mapping[label]
    for label in y_train[train_mask]
])

y_val8 = np.array([
    label_mapping[label]
    for label in y_val[val_mask]
])

print("Train:", x_train8.shape, y_train8.shape)
print("Validation:", x_val8.shape, y_val8.shape)
print("Новые метки:", np.unique(y_train8))

display(pd.DataFrame({
    "Класс": [class_names[i] for i in kept_classes],
    "Исходная метка": kept_classes,
    "Новая метка": range(8)
}))

### Обучаем на 8 классах

In [ ]:
model8 = Sequential([
    Input(shape=(32, 32, 3)),

    Rescaling(1.0 / 255),

    Conv2D(32, (3, 3), padding="same", activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), padding="same", activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), padding="same", activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(name="flatten"),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(8, activation="softmax")
])

model8.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model8.summary()

In [ ]:
checkpoint8 = ModelCheckpoint(
    filepath="best_cnn8.weights.h5",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

logger8 = CSVLogger("cnn8_training.csv")

history8 = model8.fit(
    x_train8,
    y_train8,
    validation_data=(x_val8, y_val8),
    epochs=30,
    batch_size=64,
    callbacks=[checkpoint8, logger8]
)

# Восстанавливаем лучшие веса и сохраняем модель
model8.load_weights("best_cnn8.weights.h5")
model8.save("cnn8_cifar10.keras")

In [ ]:
model8.load_weights("best_cnn8.weights.h5")

probabilities8 = model8.predict(x_test, batch_size=64)

# Предсказания в новой нумерации: 0–7
pred8_new = np.argmax(probabilities8, axis=1)

# Возвращаем исходные номера классов CIFAR-10
pred8 = np.array(kept_classes)[pred8_new]

accuracy_8 = []

for class_id in range(10):
    mask = y_test == class_id

    class_accuracy = np.mean(pred8[mask] == y_test[mask]) * 100

    accuracy_8.append(class_accuracy)

results["Ini_8class"] = accuracy_8

display(results.round(2))

# Общая точность только на восьми знакомых классах
test_mask8 = np.isin(y_test, kept_classes)

test_accuracy8 = np.mean(pred8[test_mask8] == y_test[test_mask8])

print(f"Точность на восьми классах: {test_accuracy8:.2%}")

results.to_csv(
    "comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from keras import Model

# Извлекаем обученную свёрточную часть вместе с Flatten
feature_extractor = Model(
    inputs=model8.inputs[0],
    outputs=model8.get_layer("flatten").output,
    name="frozen_features"
)

# Запрещаем изменять её веса
feature_extractor.trainable = False

# Добавляем новые полносвязные слои
transfer_model = Sequential([
    Input(shape=(32, 32, 3)),

    feature_extractor,

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(10, activation="softmax")
])

# Компилируем после заморозки
transfer_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

transfer_model.summary(expand_nested=True)

### Обучаем на 10 классах

In [ ]:
checkpoint_transfer = ModelCheckpoint(
    filepath="best_transfer.weights.h5",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

logger_transfer = CSVLogger("transfer_training.csv")

# Запоминаем веса замороженной части для проверки
frozen_weights_before = [
    weights.copy()
    for weights in feature_extractor.get_weights()
]

history_transfer = transfer_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[checkpoint_transfer, logger_transfer]
)

# Восстанавливаем лучшие веса
transfer_model.load_weights("best_transfer.weights.h5")
transfer_model.save("transfer_cifar10.keras")

# Проверяем, что замороженные веса действительно не изменились
frozen_unchanged = all(
    np.array_equal(before, after)
    for before, after in zip(
        frozen_weights_before,
        feature_extractor.get_weights()
    )
)

print("Замороженные веса не изменились:", frozen_unchanged)

In [ ]:
probabilities_transfer = transfer_model.predict(
    x_test,
    batch_size=64
)

pred_transfer = np.argmax(probabilities_transfer, axis=1)

accuracy_transfer = []

for class_id in range(10):
    mask = y_test == class_id

    class_accuracy = np.mean(pred_transfer[mask] == y_test[mask]) * 100

    accuracy_transfer.append(class_accuracy)

results["TransferLearning_10class"] = accuracy_transfer

display(results.round(2))

test_accuracy_transfer = np.mean(pred_transfer == y_test)

print(f"Тестовая точность после переноса: "
    f"{test_accuracy_transfer:.2%}"
)

results.to_csv(
    "comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

В ходе лабораторной работы была построена свёрточная нейронная сеть для классификации изображений CIFAR-10. Точность исходной модели на тестовой выборке составила 75.18%. Графики обучения показали переобучение: при дальнейшем улучшении обучающих показателей валидационная точность перестала существенно расти, а валидационные потери увеличивались.

Для исследования переноса обучения была обучена модель на восьми классах с исключением классов cat и ship. Её точность на тестовых изображениях восьми оставшихся классов составила 78.85%.

После заморозки свёрточной части и обучения новых полносвязных слоёв на десяти классах тестовая точность составила 73.91%, что на 1.27 процентного пункта ниже результата исходной модели. При этом точность распознавания кошек составила 55.6%, кораблей — 79.0%.

Таким образом, эксперимент продемонстрировал возможность повторного использования выученных признаков для дополнительных классов. В данном запуске перенос обучения с замороженной свёрточной частью не обеспечил повышения общей точности.